# HS4 Code Prediction
**Ayesha Ansari | NLP Portfolio Project**

Predict the three highest-ranked four-digit HS headings from a product description using word and character TF-IDF features with a calibrated Linear SVM.

**Workflow:** load data → extract labels → clean → split → train → evaluate → analyze errors → export.



## 1. Install dependencies and import libraries
Package versions match the project's training environment. If Colab requests a restart after installation, restart the session and run from the top. Set a fixed random seed for splitting and model fitting.

In [1]:
%pip install -q scikit-learn==1.6.1 numpy==2.1.3 pandas==2.2.3 joblib==1.6.0 datasets==4.8.5

In [2]:
# ============================================================
# FINAL HS4 CODE CLASSIFIER
# Word + Character TF-IDF + Calibrated Linear SVM
# ============================================================

import ast
import os
import re
import joblib
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report



import json
import platform
from pathlib import Path
import sklearn
import datasets
from sklearn.base import clone
from sklearn.metrics import f1_score

assert sklearn.__version__ == "1.6.1", "Restart Colab after installing dependencies."
MODEL_PATH = Path("final_hs4_calibrated_svm.pkl")
OUTPUT_DIR = Path("hs4_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load the dataset
Use the source training split of the [CROSS Rulings HTS dataset](https://huggingface.co/datasets/Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs). A fixed dataset revision supports reproducibility.

**Attribution:** CROSS Rulings HTS Dataset for Tariff Classification by Flexify.AI Inc. (https://www.flexify.ai). Review the dataset's terms before redistributing its content.

In [3]:
DATASET_REVISION = "0a66ad345d3fc9828f362836d86b31e20798513c"
# ============================================================
# 1. LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.DataFrame(
    load_dataset(
        "Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs",
        revision=DATASET_REVISION
    )["train"]
)

print("Original shape:", df.shape)

Loading dataset...


README.md:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 23.7MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/260k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/257k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Original shape: (18254, 1)


## 3. Extract labels and clean the data
Use the user message as the product description and extract an HS4 label from the assistant response. Remove invalid records, conflicting exact descriptions and duplicate text–label pairs. Retain headings with at least ten examples.

The parser selects the first dotted tariff code, with a standalone four-digit fallback. Multi-product answers and incidental numbers can introduce label noise. Source labels are not independently verified. Conflict removal and class filtering define the supported-class dataset before splitting.

In [4]:
# ============================================================
# 2. EXTRACT USER TEXT + HS4 CODE
# ============================================================

def extract_text_and_code(message):

    try:
        data = ast.literal_eval(str(message))
    except:
        data = []

    user_text = ""
    assistant_text = ""

    if isinstance(data, list):
        for m in data:
            if m.get("role") == "user" and not user_text:
                user_text = m.get("content", "")

            elif m.get("role") == "assistant" and not assistant_text:
                assistant_text = m.get("content", "")

    # Remove question wording
    user_text = re.sub(
        r"^(what\s+is\s+the\s+hts\s+(us\s+)?code\s+for\s+|"
        r"what\s+is\s+the\s+hs\s+code\s+for\s+)",
        "",
        user_text,
        flags=re.IGNORECASE
    ).rstrip("?")

    # Extract HS4 from assistant answer
    match = re.search(
        r"\b(\d{4})\.\d{2}",
        assistant_text
    )

    if not match:
        match = re.search(
            r"\b(\d{4})\b",
            assistant_text
        )

    hs4 = match.group(1) if match else None

    return user_text.lower().strip(), hs4


parsed = df["messages"].apply(
    extract_text_and_code
)

df["clean_text"] = parsed.apply(
    lambda x: x[0]
)

df["hts_code"] = parsed.apply(
    lambda x: x[1]
)

# ============================================================
# 3. CLEAN DATA
# ============================================================

df = df.dropna(
    subset=["clean_text", "hts_code"]
).copy()

# Extract first 4 digits
df["hs4"] = (
    df["hts_code"]
    .astype(str)
    .str.replace(
        r"[^\d]",
        "",
        regex=True
    )
    .str[:4]
)

# Basic text cleaning
df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep valid HS4 and meaningful text
df = df[
    df["hs4"].str.fullmatch(r"\d{4}") &
    (df["clean_text"].str.len() > 10)
].copy()


# ------------------------------------------------------------
# Remove identical text with different HS4 labels
# ------------------------------------------------------------

conflicts = (
    df.groupby("clean_text")["hs4"]
    .nunique()
)

ambiguous_text = conflicts[
    conflicts > 1
].index

df = df[
    ~df["clean_text"].isin(ambiguous_text)
].copy()


# ------------------------------------------------------------
# Remove exact duplicate examples
# ------------------------------------------------------------

df = df.drop_duplicates(
    subset=["clean_text", "hs4"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Remove very rare HS4 classes
# ------------------------------------------------------------

MIN_SAMPLES = 10

valid_classes = (
    df["hs4"]
    .value_counts()
    .loc[
        lambda x: x >= MIN_SAMPLES
    ]
    .index
)

df = df[
    df["hs4"].isin(valid_classes)
].copy()

print("\nFinal dataset shape:", df.shape)
print("HS4 classes:", df["hs4"].nunique())


Final dataset shape: (14880, 4)
HS4 classes: 252


## 4. Split the data
Create a stratified 80/20 training and holdout split with random seed 42. Check expected dataset counts, exact-text separation and enough examples for three-fold calibration.

The holdout has been inspected during project development; results are development estimates rather than an independent external benchmark. Exact deduplication does not prevent related product or ruling descriptions from appearing in both splits.

In [5]:
# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["hs4"],
    test_size=0.20,
    random_state=42,
    stratify=df["hs4"]
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
assert len(df) == 14880 and df["hs4"].nunique() == 252, "Dataset changed; stop and inspect."
assert len(X_train) == 11904 and len(X_test) == 2976
assert set(X_train).isdisjoint(set(X_test))
assert y_train.value_counts().min() >= 3
print("Expected counts and exact-text separation confirmed.")
pd.concat([
    pd.DataFrame({"record_index": X_train.index, "split": "train"}),
    pd.DataFrame({"record_index": X_test.index, "split": "development_holdout"})
]).to_csv(OUTPUT_DIR / "split_membership.csv", index=False)

Training samples: 11904
Testing samples : 2976
Expected counts and exact-text separation confirmed.


## 5. Train the model
Combine word TF-IDF (1–2 word sequences) and character TF-IDF (3–5 character sequences), then classify with a class-weighted Linear SVM.

Place the **complete feature and classifier pipeline inside `CalibratedClassifierCV`**. Each calibration fold learns its own vocabulary and IDF weights. Train using `X_train` and `y_train` only; the outer holdout is excluded from fitting.

In [6]:
base_pipeline = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            ),
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            )
        ])
    ),
    (
        "svm",
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        )
    )
])

model = CalibratedClassifierCV(
    estimator=base_pipeline,
    cv=3,
    method="sigmoid",
    n_jobs=-1
)

print("Training calibrated word + character SVM...")
model.fit(X_train, y_train)
print("Training complete.")
print("Supported HS4 headings:", len(model.classes_))


Training calibrated word + character SVM...
Training complete.
Supported HS4 headings: 252


## 6. Evaluate and export predictions
Report Top-1 accuracy, Top-3 accuracy, Macro F1 and a majority-class baseline learned from training labels. Save the classification report, all holdout predictions and incorrect Top-1 predictions.

Top-3 means the target appears among three candidates; it is not first-prediction accuracy. Macro F1 weights each heading equally.

In [7]:
# ============================================================
# MODEL EVIDENCE 1: PERFORMANCE AND BASELINE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score

# Generate predictions from the trained model
classes = np.asarray(model.classes_).astype(str)
y_true = np.asarray(y_test).astype(str)
probabilities = model.predict_proba(X_test)

# Top-1 predictions
top1_index = np.argmax(probabilities, axis=1)
predictions = classes[top1_index]

# Top-3 predictions
top3_index = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]
top3_predictions = classes[top3_index]

# Evaluate performance
top1_accuracy = accuracy_score(y_true, predictions)

top3_accuracy = np.mean([
    actual in candidates
    for actual, candidates in zip(y_true, top3_predictions)
])

macro_f1 = f1_score(
    y_true,
    predictions,
    labels=classes,
    average="macro",
    zero_division=0
)

# Majority-class baseline: always predict the most common training label
majority_code = str(y_train.astype(str).value_counts().idxmax())

baseline_predictions = np.full(
    len(y_true),
    majority_code
)

baseline_accuracy = accuracy_score(
    y_true,
    baseline_predictions
)

correct_count = int(np.sum(predictions == y_true))
wrong_count = int(np.sum(predictions != y_true))

print("MODEL EVALUATION")
print("=" * 40)
print(f"Top-1 accuracy:       {top1_accuracy:.2%}")
print(f"Top-3 accuracy:       {top3_accuracy:.2%}")
print(f"Macro F1:             {macro_f1:.4f}")
print(f"Majority class:       {majority_code}")
print(f"Majority baseline:    {baseline_accuracy:.2%}")
print(f"Test samples:         {len(y_true)}")
print(f"Correct Top-1:        {correct_count}")
print(f"Incorrect Top-1:      {wrong_count}")

# Save reproducible summary
evidence_summary = pd.DataFrame({
    "Metric": [
        "Top-1 Accuracy",
        "Top-3 Accuracy",
        "Macro F1",
        "Majority-Class Baseline",
        "Test Samples",
        "Correct Top-1",
        "Incorrect Top-1"
    ],
    "Value": [
        top1_accuracy,
        top3_accuracy,
        macro_f1,
        baseline_accuracy,
        len(y_true),
        correct_count,
        wrong_count
    ]
})

evidence_summary.to_csv(
    OUTPUT_DIR / "model_evidence_summary.csv",
    index=False
)

print("\nSaved model_evidence_summary.csv")
report = pd.DataFrame(classification_report(
    y_true, predictions, output_dict=True, zero_division=0
)).T
report.to_csv(OUTPUT_DIR / "classification_report.csv")
results_df = pd.DataFrame({
    "text": X_test.to_numpy(), "actual_hs4": y_true,
    "predicted_hs4": predictions, "top1_correct": predictions == y_true,
    "top3_predictions": [" | ".join(row) for row in top3_predictions],
    "confidence": probabilities.max(axis=1) * 100
})
results_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
results_df.loc[~results_df["top1_correct"]].to_csv(
    OUTPUT_DIR / "wrong_predictions.csv", index=False
)

MODEL EVALUATION
Top-1 accuracy:       69.83%
Top-3 accuracy:       81.59%
Macro F1:             0.5877
Majority class:       6110
Majority baseline:    7.09%
Test samples:         2976
Correct Top-1:        2078
Incorrect Top-1:      898

Saved model_evidence_summary.csv


## 7. Dataset funnel and coverage
Record retention measures how many deduplicated eligible records survive rare-class filtering. Distinct-code coverage measures the fraction of parser-derived source headings represented in the model. Neither establishes coverage of the entire HS system.

In [8]:
# ============================================================
# COVERAGE FUNNEL: RECONCILE DATASET COUNTS
# ============================================================

import pandas as pd
import numpy as np
import re

funnel_df = pd.DataFrame(
    parsed.tolist(),
    columns=["text", "raw_code"]
)

raw_count = len(funnel_df)

# Clean text and extract HS4
funnel_df["text"] = (
    funnel_df["text"].astype(str).str.lower().str.strip()
)

funnel_df["hs4"] = (
    funnel_df["raw_code"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str[:4]
)

# Remove missing, invalid, and short records
funnel_df = funnel_df[
    funnel_df["hs4"].str.fullmatch(r"\d{4}") &
    (funnel_df["text"].str.len() > 10) &
    funnel_df["raw_code"].notna()
].copy()

valid_text_count = len(funnel_df)

# Remove descriptions associated with conflicting HS4 labels
conflicts = funnel_df.groupby("text")["hs4"].nunique()
conflicting_text = conflicts[conflicts > 1].index

funnel_df = funnel_df[
    ~funnel_df["text"].isin(conflicting_text)
].copy()

after_conflicts = len(funnel_df)

# Remove exact duplicate text-label pairs
funnel_df = funnel_df.drop_duplicates(
    subset=["text", "hs4"]
).reset_index(drop=True)

after_deduplication = len(funnel_df)

# Save the pool before the rare-class filter
pre_filter_df = funnel_df.copy()

# Apply the same minimum-class threshold used in training
class_counts = funnel_df["hs4"].value_counts()
valid_classes = class_counts[class_counts >= 10].index

final_df = funnel_df[
    funnel_df["hs4"].isin(valid_classes)
].copy()

# Coverage among the deduplicated records before rare-class filtering
supported_codes = set(map(str, model.classes_))
supported_mask = pre_filter_df["hs4"].isin(supported_codes)

record_coverage = supported_mask.mean()

# Distinct codes identified in the parsed source
all_parsed_codes = set(
    pd.Series(parsed.tolist()).apply(
        lambda x: re.sub(r"\D", "", str(x[1]))[:4]
        if x[1] is not None else None
    ).dropna()
)

all_parsed_codes = {
    code for code in all_parsed_codes
    if re.fullmatch(r"\d{4}", code)
}

distinct_coverage = (
    len(all_parsed_codes & supported_codes)
    / len(all_parsed_codes)
)

funnel = pd.DataFrame({
    "Stage": [
        "Original parsed training records",
        "Valid codes and descriptions",
        "After conflicting descriptions removed",
        "After exact duplicate removal",
        "After rare-class filter"
    ],
    "Records": [
        raw_count,
        valid_text_count,
        after_conflicts,
        after_deduplication,
        len(final_df)
    ]
})

print("DATASET FUNNEL")
print(funnel.to_string(index=False))

print("\nCOVERAGE AFTER DEDUPLICATION")
print("Eligible records before rare-class filter:", len(pre_filter_df))
print("Records with supported labels:", int(supported_mask.sum()))
print(f"Record-level coverage: {record_coverage:.2%}")
print("Distinct source codes:", len(all_parsed_codes))
print("Supported distinct source codes:",
      len(all_parsed_codes & supported_codes))
print(f"Distinct-code coverage: {distinct_coverage:.2%}")

print("\nMatches existing final dataset:", len(final_df) == len(df))

funnel.to_csv(OUTPUT_DIR / "coverage_funnel.csv", index=False)

DATASET FUNNEL
                                 Stage  Records
      Original parsed training records    18254
          Valid codes and descriptions    18233
After conflicting descriptions removed    17509
         After exact duplicate removal    15697
               After rare-class filter    14880

COVERAGE AFTER DEDUPLICATION
Eligible records before rare-class filter: 15697
Records with supported labels: 14880
Record-level coverage: 94.80%
Distinct source codes: 548
Supported distinct source codes: 252
Distinct-code coverage: 45.99%

Matches existing final dataset: True


## 8. Probability reliability
Compare mean maximum model score with observed Top-1 accuracy in five bins. This top-label ECE is a descriptive diagnostic and depends on binning. Calibration does not guarantee reliable probabilities.

No deployment threshold is selected from these holdout outcomes. Scores do not reliably identify whether the true class is missing from training.

In [9]:
# ============================================================
# MODEL EVIDENCE 3: CONFIDENCE AND CALIBRATION
# ============================================================

# The highest predicted probability is the Top-1 confidence
confidence = probabilities.max(axis=1)
correct = predictions == y_true

reliability_df = pd.DataFrame({
    "confidence": confidence,
    "correct": correct
})

# Group confidence into five ranges
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.000001]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

reliability_df["confidence_bin"] = pd.cut(
    reliability_df["confidence"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

reliability_table = reliability_df.groupby(
    "confidence_bin",
    observed=False
).agg(
    Samples=("correct", "size"),
    Mean_Confidence=("confidence", "mean"),
    Actual_Accuracy=("correct", "mean")
).reset_index()

# Exclude empty bins from the calibration-error calculation
nonempty = reliability_table[
    reliability_table["Samples"] > 0
].copy()

ece = np.sum(
    nonempty["Samples"] / len(reliability_df)
    * (
        nonempty["Mean_Confidence"]
        - nonempty["Actual_Accuracy"]
    ).abs()
)

# Convert table values to percentages
reliability_table["Mean_Confidence"] *= 100
reliability_table["Actual_Accuracy"] *= 100

print("CONFIDENCE RELIABILITY")
print("=" * 40)
print(reliability_table.to_string(index=False))
print(f"\nExpected Calibration Error (ECE): {ece:.4f}")


reliability_table.to_csv(OUTPUT_DIR / "confidence_reliability.csv", index=False)
print("No deployment threshold selected for this model.")

CONFIDENCE RELIABILITY
confidence_bin  Samples  Mean_Confidence  Actual_Accuracy
         0-20%      418        12.658831        22.248804
        20-40%      599        30.204529        47.245409
        40-60%      704        50.078294        75.852273
        60-80%     1116        71.495259        92.473118
       80-100%      139        81.408800        97.841727

Expected Calibration Error (ECE): 0.1951
No deployment threshold selected for this model.


## 9. Similarity and generalization
Fit a diagnostic TF-IDF representation using training text only. Find each holdout description's nearest training description and summarize accuracy by cosine similarity.

High similarity warrants inspection but does not alone prove leakage. Low similarity can expose weak performance on unfamiliar wording. This diagnostic does not remove cases or tune the model.

In [10]:
diagnostic_features = clone(base_pipeline.named_steps["features"])
X_train_features = diagnostic_features.fit_transform(X_train)
X_test_features = diagnostic_features.transform(X_test)

# ============================================================
# NEAR-DUPLICATE SIMILARITY CHECK
# ============================================================

from sklearn.neighbors import NearestNeighbors

nearest_model = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nearest_model.fit(X_train_features)

distances, indices = nearest_model.kneighbors(X_test_features)

similarity = np.clip(1 - distances.ravel(), 0, 1)

actual = np.asarray(y_test).astype(str)

combined_probs = model.predict_proba(X_test)
combined_classes = model.classes_.astype(str)

top1 = combined_classes[np.argmax(combined_probs, axis=1)]
top3 = combined_classes[
    np.argsort(combined_probs, axis=1)[:, -3:][:, ::-1]
]

similarity_df = pd.DataFrame({
    "test_text": X_test.to_numpy(),
    "actual_hs4": actual,
    "predicted_hs4": top1,
    "top1_correct": top1 == actual,
    "top3_correct": [
        label in candidates
        for label, candidates in zip(actual, top3)
    ],
    "nearest_train_text": X_train.iloc[indices.ravel()].to_numpy(),
    "nearest_similarity": similarity
})

bins = [0, 0.25, 0.50, 0.70, 0.85, 1.000001]
labels = ["0–0.25", "0.25–0.50", "0.50–0.70",
          "0.70–0.85", "0.85–1.00"]

similarity_df["similarity_band"] = pd.cut(
    similarity_df["nearest_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

report = similarity_df.groupby(
    "similarity_band",
    observed=False
).agg(
    samples=("top1_correct", "size"),
    mean_similarity=("nearest_similarity", "mean"),
    top1_accuracy=("top1_correct", "mean"),
    top3_accuracy=("top3_correct", "mean")
).reset_index()

for column in ["mean_similarity", "top1_accuracy", "top3_accuracy"]:
    report[column] *= 100

print("ACCURACY BY NEAREST-TRAINING SIMILARITY")
print(report.to_string(index=False))

print("\nHigh-similarity test cases (similarity >= 0.90):")

high = similarity_df["nearest_similarity"] >= 0.90
print("Cases:", int(high.sum()))

if high.any():
    print("Top-1 accuracy:",
          f"{similarity_df.loc[high, 'top1_correct'].mean():.2%}")
    print("Top-3 accuracy:",
          f"{similarity_df.loc[high, 'top3_correct'].mean():.2%}")

similarity_df.to_csv(OUTPUT_DIR / "near_duplicate_analysis.csv", index=False)

ACCURACY BY NEAREST-TRAINING SIMILARITY
similarity_band  samples  mean_similarity  top1_accuracy  top3_accuracy
         0–0.25       87        22.639584      39.080460      54.022989
      0.25–0.50     1482        38.293135      59.311741      73.684211
      0.50–0.70      739        59.079924      79.161028      90.527740
      0.70–0.85      369        77.454463      86.720867      92.411924
      0.85–1.00      299        92.351471      86.956522      93.311037

High-similarity test cases (similarity >= 0.90):
Cases: 196
Top-1 accuracy: 87.24%
Top-3 accuracy: 93.37%


## 10. Illustrative predictions
Explore the model's candidate rankings for several descriptions. These examples are not an independent benchmark, and their wording may overlap source data. Frother and bottle reference labels have not been independently verified. Verify all suggested classifications.

In [11]:
examples = {
    "Bralette": (
        "A girl's bralette made of 95% cotton and 5% spandex, "
        "with adjustable shoulder straps and a hook-and-eye back closure"
    ),
    "Pacemaker leads": "Pacing leads used in pacemakers",
    "Water descaler": (
        "A high-speed pressure water descaler used in rolling mills "
        "to remove scale from seamless steel pipes"
    ),
    "Woven dress": (
        "Women's 100% woven cotton summer dress with floral embroidery"
    ),
    "Milk frother": (
        "A rechargeable handheld milk frother with a small electric "
        "motor and stainless steel whisk, used to foam milk for coffee"
    ),
    "Vacuum bottle": (
        "An insulated reusable drinking bottle with stainless steel "
        "inner and outer walls, a vacuum between the walls, "
        "and a screw-on lid"
    )
}

for name, text in examples.items():
    probabilities = model.predict_proba(
        [text.lower().strip()]
    )[0]

    indices = np.argsort(probabilities)[::-1][:3]

    print(f"\n{name}")
    for index in indices:
        print(
            f"HS4 {model.classes_[index]}: "
            f"{probabilities[index]:.2%}"
        )


Bralette
HS4 6108: 27.05%
HS4 6106: 19.09%
HS4 4202: 7.98%

Pacemaker leads
HS4 9002: 13.61%
HS4 8409: 10.85%
HS4 9018: 4.98%

Water descaler
HS4 8424: 49.20%
HS4 7304: 11.84%
HS4 8423: 9.62%

Woven dress
HS4 6204: 21.47%
HS4 6104: 11.72%
HS4 6110: 4.53%

Milk frother
HS4 8516: 22.65%
HS4 8509: 13.91%
HS4 8501: 4.55%

Vacuum bottle
HS4 7323: 23.03%
HS4 3923: 6.70%
HS4 7318: 6.69%


## 11. Save and verify model artifacts
Export the newly trained model, metadata and evaluation files. Reload the exported artifact and confirm that saving preserves predictions. The model is loaded here only to verify the file **after training**, not to initialize training.

Download `hs4_project_outputs.zip` through Colab's Files sidebar. Save the executed notebook with **File → Download → Download .ipynb**.

In [12]:
model_info = {
    "model_type": "Word + Character TF-IDF + Calibrated Linear SVM",
    "model_file": MODEL_PATH.name, "target": "HS4",
    "num_classes": len(model.classes_), "classes": model.classes_.tolist(),
    "training_samples": len(X_train), "test_samples": len(X_test),
    "top1_accuracy": float(top1_accuracy), "top3_accuracy": float(top3_accuracy),
    "macro_f1": float(macro_f1), "majority_baseline": float(baseline_accuracy),
    "calibration": "sigmoid", "calibration_cv": 3,
    "tfidf_fitted_within_calibration_folds": True,
    "review_threshold": None,
    "evaluation_note": "Previously inspected development holdout; not an untouched final benchmark.",
    "dataset_id": "Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs",
    "dataset_revision": DATASET_REVISION,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
    "pandas_version": pd.__version__, "joblib_version": joblib.__version__,
    "datasets_version": datasets.__version__
}
export_path = OUTPUT_DIR / MODEL_PATH.name
joblib.dump(model, export_path, compress=3)
joblib.dump(model_info, OUTPUT_DIR / "model_info.pkl")
(OUTPUT_DIR / "model_info.json").write_text(json.dumps(model_info, indent=2))
reloaded = joblib.load(export_path)
np.testing.assert_allclose(
    model.predict_proba(X_train.iloc[:5]), reloaded.predict_proba(X_train.iloc[:5]),
    rtol=1e-10, atol=1e-12
)
print("Save/reload prediction check passed.")
import shutil
archive = shutil.make_archive("hs4_project_outputs", "zip", OUTPUT_DIR)
print("Download:", archive)

Save/reload prediction check passed.
Download: /content/hs4_project_outputs.zip


## Limitations
- Source-answer parsing can introduce incorrect or ambiguous labels, especially for multiple products or codes.
- Classes with fewer than ten examples are excluded. The model cannot predict unsupported headings.
- Exact duplicates are removed, but related products or rulings may occur across the random split.
- The holdout was inspected during development and is not an untouched final benchmark.
- Performance on unfamiliar descriptions is uneven, and probability estimates may remain poorly calibrated.
- No validated review threshold or missing-class detector is established.
- HS4 headings are not complete country-specific tariff codes. This is an educational candidate-ranking prototype requiring human verification.